# Chapter 15 — Memory Search as a Random Walk

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/15_memory_search/).

Abbott, Austerweil & Griffiths (2012): human semantic fluency is a **censored random walk** on a semantic network. We build a small network, sample walks, apply the censoring function, compute IRTs, and watch the position-1-slowest signature emerge — with no switch rule.

## Setup

In [ ]:
!pip install genjax -q
print('✅ ready')

## The network

Two animal clusters (pets, big animals) joined by a bridge (tiger), plus non-animal distractors and a cue node.

In [ ]:
import jax.numpy as jnp
import numpy as np

names  = ["dog", "cat", "hamster", "tiger", "lion", "zebra", "giraffe",
          "house", "food", "water", "animal"]
is_animal = np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
word_len  = np.array([3, 3, 7, 5, 4, 5, 7, 5, 4, 5, 6])
idx = {n: i for i, n in enumerate(names)}
N = len(names)

edges = [
    ("dog", "cat"), ("dog", "hamster"), ("cat", "hamster"),
    ("lion", "zebra"), ("lion", "giraffe"), ("zebra", "giraffe"),
    ("cat", "tiger"), ("tiger", "lion"),
    ("dog", "house"), ("house", "food"), ("food", "water"),
    ("water", "cat"), ("hamster", "food"),
    ("animal", "dog"), ("animal", "lion"),
]
L = np.zeros((N, N))
for a, b in edges:
    L[idx[a], idx[b]] = 1.0
    L[idx[b], idx[a]] = 1.0
L = jnp.array(L)

degree = L.sum(axis=1)
P = L / degree[:, None]
print("network:", N, "nodes,", int(L.sum() // 2), "edges")

## Sample many walks (batched)

In [ ]:
import jax
import jax.random as jr

LOGP = jnp.log(jnp.where(P > 0, P, 1e-30))
STEPS = 400

def one_walk(key):
    def step(node, k):
        nxt = jr.categorical(k, LOGP[node])
        return nxt, nxt
    _, visited = jax.lax.scan(step, idx["animal"], jr.split(key, STEPS))
    return visited

walk_batch = jax.jit(jax.vmap(one_walk))
walks = np.array(walk_batch(jr.split(jr.key(0), 500)))
print("sampled", walks.shape[0], "walks of", walks.shape[1], "steps each")

## Censor, compute IRTs, find the signature

Keep each animal's first visit; IRT = wander-time between first-hits + word length; label by position within the patch. Position 1 (first word of a new cluster) should be slowest.

In [ ]:
from collections import defaultdict

cluster = {"dog": 0, "cat": 0, "hamster": 0,
           "lion": 1, "zebra": 1, "giraffe": 1, "tiger": 2}

def censored_irts(walk):
    seen, taus, words = set(), [], []
    for t, node in enumerate(walk):
        if is_animal[node] and node not in seen:
            seen.add(node); taus.append(t); words.append(node)
    irts = [taus[k] - taus[k-1] + int(word_len[words[k]])
            for k in range(1, len(words))]
    return words, irts

def patch_positions(words):
    cl = [cluster[names[w]] for w in words]
    pos, run = [], 1
    for k in range(1, len(words)):
        run = 1 if (cl[k] != cl[k-1] and 2 not in (cl[k], cl[k-1])) else run + 1
        pos.append(run)
    return pos

by_position = defaultdict(list)
for walk in walks:
    words, irts = censored_irts(walk)
    if len(irts) < 3:
        continue
    for p, irt in zip(patch_positions(words), irts):
        if p <= 3:
            by_position[p].append(irt)

avg = np.mean([x for v in by_position.values() for x in v])
print(f"average IRT over all positions: {avg:.1f}")
for p in [1, 2, 3]:
    ratio = np.mean(by_position[p]) / avg
    print(f"patch position {p}: IRT / average = {ratio:.2f}")

## Try it yourself

1. Delete the bridge edges (`cat–tiger`, `tiger–lion`) so the clusters disconnect. Re-run — can one walk now report from both clusters? What happens to the position-1 signature?
2. Add more distractor nodes between the clusters (a longer bridge path). Does the position-1 IRT ratio go up or down? Relate to the τ(k) − τ(k−1) term.